In [81]:
from functools import partial 
print = partial(print, flush=True) 

In [82]:
import numpy as np, pandas as pd 
import matplotlib.pyplot as plt 
from matplotlib.pyplot import subplots 
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import KFold, train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report 
from sklearn.pipeline import Pipeline 

from ISLP import load_data 
from ISLP.models import ModelSpec as MS 
from ISLP.torch import ErrorTracker, SimpleDataModule, SimpleModule 

import torch 
from torch import nn 
from torch.optim import RMSprop 
from torch.utils.data import TensorDataset 
from torchmetrics import MeanAbsoluteError, R2Score 
from torchinfo import summary 
from pytorch_lightning import Trainer 
from pytorch_lightning.loggers import CSVLogger

import json 
from glob import glob 
import seaborn as sns 


In [83]:
%matplotlib inline 
sns.set_theme()

from pytorch_lightning import seed_everything 
seed_everything(1, workers=True) 
torch.use_deterministic_algorithms(True, warn_only=True) 

Seed set to 1


In [84]:
default = load_data("Default") 
print(default.shape) 
default.head(5)

(10000, 4)


,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [85]:
model = MS(default.columns.drop("default"), intercept=False) 
X = model.fit_transform(default).to_numpy() 
Y = (default["default"] == "Yes").to_numpy(dtype=int)


In [86]:
X_train, X_test, Y_train, Y_test = train_test_split( 
X, Y, test_size=0.3, random_state=1 
) 


In [87]:
scaler = StandardScaler() 
X_train[:, 1:] = scaler.fit_transform(X_train[:, 1:]) 
X_test[:, 1:] = scaler.transform(X_test[:, 1:])


In [88]:
class DefaultModel(nn.Module): 
    def __init__(self, input_size, num_hidden_nodes): 
        super(DefaultModel, self).__init__() 
        self.flatten = nn.Flatten() 
        self.hidden_layer = nn.Sequential(nn.Linear(input_size,num_hidden_nodes), nn.ReLU(), nn.Dropout(0.2)) 
        self.output_layer = nn.Sequential(nn.Linear(num_hidden_nodes, 2), nn.Softmax(dim=1)) 
        
    def forward(self, x): 
        x = self.flatten(x) 
        x = self.hidden_layer(x) 
        x = self.output_layer(x) 
        return x 
    
default_model = DefaultModel(X.shape[1], num_hidden_nodes=10)


In [89]:
print(default_model) 
summary( 
    default_model, 
    input_size=X_train.shape, 
    col_names=["input_size", "output_size", "num_params"], 
)


DefaultModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden_layer): Sequential(
    (0): Linear(in_features=3, out_features=10, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (output_layer): Sequential(
    (0): Linear(in_features=10, out_features=2, bias=True)
    (1): Softmax(dim=1)
  )
)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
DefaultModel                             [7000, 3]                 [7000, 2]                 --
├─Flatten: 1-1                           [7000, 3]                 [7000, 3]                 --
├─Sequential: 1-2                        [7000, 3]                 [7000, 10]                --
│    └─Linear: 2-1                       [7000, 3]                 [7000, 10]                40
│    └─ReLU: 2-2                         [7000, 10]                [7000, 10]                --
│    └─Dropout: 2-3                      [7000, 10]                [7000, 10]                --
├─Sequential: 1-3                        [7000, 10]                [7000, 2]                 --
│    └─Linear: 2-4                       [7000, 10]                [7000, 2]                 22
│    └─Softmax: 2-5                      [7000, 2]                 [7000, 2]                 --
Total params: 62
Trainable params: 

In [90]:
X_train_t = torch.tensor(X_train, dtype=torch.float32) 
Y_train_t = torch.tensor(Y_train, dtype=torch.long) 
default_train = TensorDataset(X_train_t, Y_train_t) 
X_test_t = torch.tensor(X_test, dtype=torch.float32) 
Y_test_t = torch.tensor(Y_test, dtype=torch.long) 
default_test = TensorDataset(X_test_t, Y_test_t) 


In [91]:
from ISLP.torch import rec_num_workers 
max_num_workers = rec_num_workers() 
print(max_num_workers) 


12


In [92]:
default_dm = SimpleDataModule( 
default_train, default_test, batch_size=32, num_workers=max_num_workers, validation=default_test ) 


In [93]:
from torchmetrics import Accuracy 
default_module = SimpleModule( 
    default_model, 
    loss=nn.CrossEntropyLoss(), 
    optimizer=torch.optim.RMSprop(default_model.parameters(), lr=0.01), 
    metrics={"val_acc": Accuracy(task="multiclass", num_classes=2)}, 
) 


In [94]:
default_logger = CSVLogger("logs", name="default") 


In [95]:
default_trainer = Trainer( 
deterministic=True, 
max_epochs=30, 
log_every_n_steps=5, 
logger=default_logger, 
callbacks=[ErrorTracker()], 
) 
default_trainer.fit(default_module, datamodule=default_dm) 


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name  | Type             | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | model | DefaultModel     | 62     | train | 0    
1 | loss  | CrossEntropyLoss | 0      | train | 0    
-----------------------------------------------------------
62        Trainable params
0         Non-trainable params
62        Total params
0.000     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\Programming\Classwork\Classwork (Repo)\STAT4210\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 29: 100%|██████████| 219/219 [00:02<00:00, 105.32it/s, v_num=4]       

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 219/219 [00:02<00:00, 104.32it/s, v_num=4]


In [96]:
default_trainer.test(default_module, datamodule=default_dm)

Testing DataLoader 0: 100%|██████████| 94/94 [00:00<00:00, 338.46it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.3373628854751587
      test_val_acc          0.9756666421890259
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.3373628854751587, 'test_val_acc': 0.9756666421890259}]

In [97]:
default_module.eval() 
with torch.no_grad(): 
    outputs = default_module.model(X_test_t) 
    preds = torch.argmax(outputs, dim=1).numpy() 
    np.unique(preds, return_counts=True) 
    from sklearn.metrics import confusion_matrix 
    cm = confusion_matrix(Y_test, preds) 
    print(cm)


[[2899   10]
 [  63   28]]


In [98]:
lr = LogisticRegression() 
lr.fit(X_train, Y_train) 
preds = lr.predict(X_test) 
np.unique(preds, return_counts=True) 
from sklearn.metrics import confusion_matrix 
cm = confusion_matrix(Y_test, preds) 
print(cm) 


[[2895   14]
 [  59   32]]


In [99]:
trainer_15 = Trainer( 
    deterministic=True, 
    max_epochs=30, 
    log_every_n_steps=5, 
    logger=default_logger, 
    callbacks=[ErrorTracker()], 
) 
model_15 = DefaultModel(X.shape[1], num_hidden_nodes=15)
module_15 = SimpleModule( 
    model_15, 
    loss=nn.CrossEntropyLoss(), 
    optimizer=torch.optim.RMSprop(model_15.parameters(), lr=0.01), 
    metrics={"val_acc": Accuracy(task="multiclass", num_classes=2)}, 
) 
trainer_15.fit(module_15, datamodule=default_dm) 

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
d:\Programming\Classwork\Classwork (Repo)\STAT4210\.venv\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory logs\default\version_4\checkpoints exists and is not empty.

  | Name  | Type             | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | model | DefaultModel     | 92     | train | 0    
1 | loss  | CrossEntropyLoss | 0      | train | 0    
-----------------------------------------------------------
92        Trainable params
0         Non-trainable params
92        Total params
0.000     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\Programming\Classwork\Classwork (Repo)\STAT4210\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 29: 100%|██████████| 219/219 [00:02<00:00, 94.93it/s, v_num=4]        

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 219/219 [00:02<00:00, 94.52it/s, v_num=4]


In [100]:
trainer_15.test(module_15, datamodule=default_dm)

Testing DataLoader 0: 100%|██████████| 94/94 [00:01<00:00, 70.77it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.3435949385166168
      test_val_acc          0.9696666598320007
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.3435949385166168, 'test_val_acc': 0.9696666598320007}]

In [101]:
module_15.eval() 
with torch.no_grad(): 
    outputs = module_15.model(X_test_t) 
    preds = torch.argmax(outputs, dim=1).numpy() 
    np.unique(preds, return_counts=True) 
    from sklearn.metrics import confusion_matrix 
    cm = confusion_matrix(Y_test, preds) 
    print(cm)

[[2909    0]
 [  91    0]]


In [102]:
trainer_20 = Trainer( 
    deterministic=True, 
    max_epochs=30, 
    log_every_n_steps=5, 
    logger=default_logger, 
    callbacks=[ErrorTracker()], 
) 
model_20 = DefaultModel(X.shape[1], num_hidden_nodes=20)
module_20 = SimpleModule( 
    model_20, 
    loss=nn.CrossEntropyLoss(), 
    optimizer=torch.optim.RMSprop(model_20.parameters(), lr=0.01), 
    metrics={"val_acc": Accuracy(task="multiclass", num_classes=2)}, 
) 
trainer_20.fit(module_20, datamodule=default_dm) 

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
d:\Programming\Classwork\Classwork (Repo)\STAT4210\.venv\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory logs\default\version_4\checkpoints exists and is not empty.

  | Name  | Type             | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | model | DefaultModel     | 122    | train | 0    
1 | loss  | CrossEntropyLoss | 0      | train | 0    
-----------------------------------------------------------
122       Trainable params
0         Non-trainable params
122       Total params
0.000     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\Programming\Classwork\Classwork (Repo)\STAT4210\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 29: 100%|██████████| 219/219 [00:02<00:00, 98.40it/s, v_num=4]        

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 219/219 [00:02<00:00, 97.97it/s, v_num=4]


In [103]:
trainer_20.test(module_20, datamodule=default_dm)

Testing DataLoader 0: 100%|██████████| 94/94 [00:01<00:00, 63.76it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.3391546905040741
      test_val_acc          0.9733333587646484
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.3391546905040741, 'test_val_acc': 0.9733333587646484}]

In [104]:
module_20.eval() 
with torch.no_grad(): 
    outputs = module_20.model(X_test_t) 
    preds = torch.argmax(outputs, dim=1).numpy() 
    np.unique(preds, return_counts=True) 
    from sklearn.metrics import confusion_matrix 
    cm = confusion_matrix(Y_test, preds) 
    print(cm)

[[2904    5]
 [  75   16]]
